In [1]:
import numpy as np
import math
import rclpy
import threading
import time

from semantic_digital_twin.world import World
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
from semantic_digital_twin.robots.soft_trunk import SoftTrunk


/home/rinaalo/.virtualenvs/cram-env/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [ ]:
#### PCC TEST ####

# Initialize ROS 2 Node
if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node("soft_pcc_test")
thread = threading.Thread(target=rclpy.spin, args=(node,), daemon=True)
thread.start()

# Create the snake with 3 sections, and 10 visual segments per section
num_sections = 5
world = World()
trunk, kappas, phis = SoftTrunk.build_pcc(world, num_sections=num_sections, segs_per_section=10, total_length=2.0)

# Setup Visualization
tf_pub = TFPublisher(_world=world, node=node)
viz_pub = VizMarkerPublisher(_world=world, node=node)
tf_pub.add_to_world(world)
viz_pub.add_to_world(world)

print(f"Robot '{trunk.name.name}' is ready. Check RViz.")


In [ ]:
# set the fixed frame to pcc/base in rviz
try:
    while True:
        print("Animating PCC")
        for t in np.linspace(0, 20, 400):
            
            for s in range(num_sections):
                # The 's * 1.0' creates a delay so section 2 starts bending 
                # slightly after section 1.
                k_val = 1.5 * np.sin(t - s * 1.0) 
                
                # make it spiral by rotating the phis
                p_val = t * 0.5 
                
                # Update the specific DOFs for this section
                world.state[kappas[s].id].position = float(k_val)
                world.state[phis[s].id].position = float(p_val)
            
            # Recompute everything
            world.notify_state_change()
            
            # Update RViz
            tf_pub.notify()
            viz_pub.notify()
            
            time.sleep(0.03)
except KeyboardInterrupt:
    print("Animation stopped.")

In [ ]:
node.destroy_node()
rclpy.shutdown()

In [2]:
### CUSTOM PCC TEST ###

if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node("custom_pcc_test")
thread = threading.Thread(target=rclpy.spin, args=(node,), daemon=True)
thread.start()

lengths = [0.2, 0.4, 0.8]
radii = [0.1, 0.05, 0.02]
resolutions = [5, 10, 20]

world = World()
trunk, kappas, phis = SoftTrunk.build_custom_pcc(world, lengths, radii, resolutions)

tf_pub = TFPublisher(_world=world, node=node)
viz_pub = VizMarkerPublisher(_world=world, node=node)
tf_pub.add_to_world(world)
viz_pub.add_to_world(world)

In [3]:
try:
    while True:
        print("Animating PCC")
        for t in np.linspace(0, 20, 400):
            for s in range(3):
                # Curvature
                k_val = 2.0 * np.sin(t * (1 + s*0.3)) 
                
                # Instead of a smooth rotation, force sections into specific planes.
                p_val = s * (np.pi / 2) 
                
                # Update the world state
                world.state[kappas[s].id].position = float(k_val)
                world.state[phis[s].id].position = float(p_val)
                
            world.notify_state_change()
            
            # Update RViz
            tf_pub.notify()
            viz_pub.notify()
            
            time.sleep(0.03)
except KeyboardInterrupt:
    print("Animation stopped.")

Animating PCC
Animating PCC
Animation stopped.


In [ ]:
# sections bend in opposite directions (180 degrees apart)
try:
    while True:
        print("Animating PCC")
        for t in np.linspace(0, 20, 400):
            for s in range(3):
                k_val = 2.5 * np.sin(t) 
                
                # alternating sections bend in opposite directions
                if s % 2 == 0:
                    p_val = 0.0 # Bend "Forward"
                else:
                    p_val = np.pi # Bend "Backward"
                    
                world.state[kappas[s].id].position = float(k_val)
                world.state[phis[s].id].position = float(p_val)
                
            world.notify_state_change()
            
            # Update RViz
            tf_pub.notify()
            viz_pub.notify()
            
            time.sleep(0.03)
except KeyboardInterrupt:
    print("Animation stopped.")

In [4]:
node.destroy_node()
rclpy.shutdown()

In [5]:
#### COSSERAT ROD TEST ####

# Initialize ROS 2 Node
if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node("cosserat_test")
thread = threading.Thread(target=rclpy.spin, args=(node,), daemon=True)
thread.start()

world = World()
num_sections = 5

trunk, ux_list, uy_list, uz_list, vz_list = SoftTrunk.build_cosserat(world, num_sections=num_sections, segs_per_section=10, total_length=2.0)

tf_pub = TFPublisher(_world=world, node=node)
viz_pub = VizMarkerPublisher(_world=world, node=node)
tf_pub.add_to_world(world)
viz_pub.add_to_world(world)

INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name cosserat/base
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name cosserat/sec0_seg0
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name cosserat/base
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name cosserat/sec0_seg0
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name cosserat/sec0_seg1
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name cosserat/sec0_seg0
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name cosserat/sec0_seg1
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name cosserat/sec0_seg2
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name cosserat/sec0_seg1
INFO:semantic_digital_twin.world:Trying to add kinematic_structure_entity with name

In [ ]:
# set the fixed frame to cosserat/base in rviz
try:
    print("Animating Cosserat")
    while True:
        for t in np.linspace(0, 10, 200):
            for s in range(num_sections):
                # Bending: Create a circular motion using ux and uy
                # Values between -2 and 2
                val_ux = 2.0 * np.sin(t)
                val_uy = 2.0 * np.cos(t)
                
                # Torsion: Twist the rod back and forth
                val_uz = 1.5 * np.sin(t * 0.5)
                
                # strecth
                stretch_val = 1.0 + 0.5 * np.sin(t)
                #stretch_val = 1

                # Update World State
                world.state[vz_list[s].id].position = float(stretch_val)
                world.state[ux_list[s].id].position = float(val_ux)
                world.state[uy_list[s].id].position = float(val_uy)
                world.state[uz_list[s].id].position = float(val_uz)                

            
            world.notify_state_change()
            tf_pub.notify()
            viz_pub.notify()
            
            time.sleep(0.05)
except KeyboardInterrupt:
    print("Stopped.")

Animating Cosserat


In [ ]:
node.destroy_node()
rclpy.shutdown()

In [2]:
#### COSSERAT ROD TEST ####

# Initialize ROS 2 Node
if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node("cosserat_test")
thread = threading.Thread(target=rclpy.spin, args=(node,), daemon=True)
thread.start()

world = World()
num_sections = 5

lengths = [0.3, 0.3, 0.3]
radii = [0.08, 0.04, 0.01]
res = [10, 10, 10]
trunk, ux_list, uy_list, uz_list, vz_list = SoftTrunk.build_custom_cosserat(world, lengths, radii, res)


tf_pub = TFPublisher(_world=world, node=node)
viz_pub = VizMarkerPublisher(_world=world, node=node)
tf_pub.add_to_world(world)
viz_pub.add_to_world(world)

In [4]:
# set the fixed frame to cosserat/base in rviz
try:
    print("Animating Cosserat")
    while True:
        for t in np.linspace(0, 10, 200):
            for s in range(3):
                # Bending: Create a circular motion using ux and uy
                # Values between -2 and 2
                val_ux = 2.0 * np.sin(t)
                val_uy = 2.0 * np.cos(t)
                
                # Torsion: Twist the rod back and forth
                val_uz = 1.5 * np.sin(t * 0.5)
                
                # strecth
                stretch_val = 1.0 + 0.5 * np.sin(t)
                #stretch_val = 1

                # Update World State
                world.state[vz_list[s].id].position = float(stretch_val)
                world.state[ux_list[s].id].position = float(val_ux)
                world.state[uy_list[s].id].position = float(val_uy)
                world.state[uz_list[s].id].position = float(val_uz)                

            
            world.notify_state_change()
            tf_pub.notify()
            viz_pub.notify()
            
            time.sleep(0.05)
except KeyboardInterrupt:
    print("Stopped.")

Animating Cosserat
Stopped.


In [5]:
node.destroy_node()
rclpy.shutdown()